# Project 1: Chatbot (Multilingual Text-to-Text System)

**Team Members:**
- Arqam Bin Adnan — FA23-BSCS-0082
- Adeel Shaikh — FA23-BSCS-0074

**Course:** Generative AI | **Section:** AW

---

### Notebook Structure
1. Setup & Installs
2. Dataset Collection (Wikipedia Multilingual)
3. Preprocessing Analysis (Raw vs Cleaned)
4. Chunking Analysis (Recursive vs Token)
5. Retrieval Model 1 — `paraphrase-multilingual-MiniLM-L12-v2`
6. Retrieval Model 2 — `intfloat/multilingual-e5-small`
7. Retrieval Model Comparison
8. Generation Model 1 — `google/gemma-3-4b-it`
9. Generation Model 2 — `Qwen/Qwen2.5-3B-Instruct`
10. Full RAG Pipeline with Language Control
11. Generation Model Comparison
12. Edge Case Testing

## Section 1 — Setup & Installs

In [ ]:
!pip install transformers datasets accelerate bitsandbytes
!pip install langchain-text-splitters sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 74.3 MB/s eta 0:00:00


In [ ]:
from google.colab import userdata
hf_token = userdata.get('HF_TOKEN_TW')

In [ ]:
from huggingface_hub import login
login(hf_token)
print('Successfully logged in to Hugging Face.')

Successfully logged in to Hugging Face.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Section 2 — Dataset Collection

Fetching university articles directly from Wikipedia's Category API across 4 languages (EN, DE, ES, NL).
**Skip this cell if data is already saved on Drive, run the load cell below instead.**

In [ ]:
# FETCH DATASET (only run once, then use load cell below)
'''
import requests, json, time

MAX_ARTICLES_PER_LANG = 200
MAX_CHARS_PER_ARTICLE = 3000
HEADERS = {"User-Agent": "GenAI-Course-Project/1.0 (university RAG chatbot; student project)"}

LANG_CATEGORIES = {
    "en": ("en", "Universities_and_colleges_by_country"),
    "de": ("de", "Universität"), #German
    "es": ("es", "Universidades"), #Spanish
    "nl": ("nl", "Universiteit"), #Dutch
}

def get_titles_from_category(lang_code, category, limit=200):
    url = f"https://{lang_code}.wikipedia.org/w/api.php"
    params = {"action": "query", "list": "categorymembers", "cmtitle": f"Category:{category}",
              "cmlimit": limit, "cmtype": "page", "format": "json"}
    for attempt in range(5):
        try:
            response = requests.get(url, params=params, headers=HEADERS, timeout=20)
            if not response.text.strip():
                wait = (attempt + 1) * 5
                print(f"  [{lang_code}] Empty response, waiting {wait}s...")
                time.sleep(wait)
                continue
            data = response.json()
            titles = [m["title"] for m in data.get("query", {}).get("categorymembers", [])]
            print(f"[{lang_code}] Found {len(titles)} articles")
            return titles
        except Exception as e:
            wait = (attempt + 1) * 5
            print(f"  [{lang_code}] Error: {e}, waiting {wait}s...")
            time.sleep(wait)
    return []

def get_article_text(lang_code, title, max_chars=3000):
    url = f"https://{lang_code}.wikipedia.org/w/api.php"
    params = {"action": "query", "titles": title, "prop": "extracts",
              "explaintext": True, "exsectionformat": "plain", "exintro": True, "format": "json"}
    for attempt in range(5):
        try:
            response = requests.get(url, params=params, headers=HEADERS, timeout=20)
            if not response.text.strip():
                time.sleep((attempt + 1) * 5)
                continue
            pages = response.json()["query"]["pages"]
            text = next(iter(pages.values())).get("extract", "")
            return text[:max_chars]
        except Exception as e:
            time.sleep((attempt + 1) * 5)
    return ""

all_texts = []
for lang_key, (lang_code, category) in LANG_CATEGORIES.items():
    titles = get_titles_from_category(lang_code, category, limit=MAX_ARTICLES_PER_LANG)
    for i, title in enumerate(titles):
        text = get_article_text(lang_code, title, max_chars=MAX_CHARS_PER_ARTICLE)
        if text.strip():
            all_texts.append({"title": title, "text": text, "lang": lang_key})
        time.sleep(5 if (i + 1) % 10 == 0 else 2)
        if (i + 1) % 10 == 0:
            print(f"  [{lang_key}] {i+1}/{len(titles)} done | {len(all_texts)} collected")

print(f"Total articles: {len(all_texts)}")
with open("/content/drive/MyDrive/uni_wiki_multilingual.json", "w", encoding="utf-8") as f:
    json.dump(all_texts, f, ensure_ascii=False)
print("Saved to Drive!")
'''

'\nimport requests, json, time\n\nMAX_ARTICLES_PER_LANG = 200\nMAX_CHARS_PER_ARTICLE = 3000\nHEADERS = {"User-Agent": "GenAI-Course-Project/1.0 (university RAG chatbot; student project)"}\n\nLANG_CATEGORIES = {\n    "en": ("en", "Universities_and_colleges_by_country"),\n    "de": ("de", "Universität"),\n    "es": ("es", "Universidades"),\n    "fr": ("fr", "Université"),\n}\n\ndef get_titles_from_category(lang_code, category, limit=200):\n    url = f"https://{lang_code}.wikipedia.org/w/api.php"\n    params = {"action": "query", "list": "categorymembers", "cmtitle": f"Category:{category}",\n              "cmlimit": limit, "cmtype": "page", "format": "json"}\n    for attempt in range(5):\n        try:\n            response = requests.get(url, params=params, headers=HEADERS, timeout=20)\n            if not response.text.strip():\n                wait = (attempt + 1) * 5\n                print(f"  [{lang_code}] Empty response, waiting {wait}s...")\n                time.sleep(wait)\n        

In [ ]:
# ── LOAD DATASET FROM DRIVE ──────────────────────────────────────────────────
import json

with open('/content/drive/MyDrive/uni_wiki_multilingual.json', 'r', encoding='utf-8') as f:
    all_texts = json.load(f)

print(f"Loaded {len(all_texts)} articles")
for lang in ['en', 'de', 'es', 'nl']:
    count = sum(1 for a in all_texts if a['lang'] == lang)
    print(f"  [{lang}] {count} articles")

Loaded 151 articles
  [en] 1 articles
  [de] 19 articles
  [es] 32 articles
  [nl] 99 articles


## Section 3 — Preprocessing Analysis

Comparing two approaches:
- **Raw**: text used as-is from Wikipedia
- **Cleaned**: lowercased, whitespace collapsed, special characters removed

In [ ]:
import re

# --- Preprocessing Method 1: Raw (no changes) ---
def preprocess_raw(text):
    return text.strip()

# --- Preprocessing Method 2: Cleaned ---
def preprocess_cleaned(text):
    text = text.lower()
    text = re.sub(r'\s+', ' ', text)                    # collapse whitespace
    text = re.sub(r'[^\w\s\u0600-\u06FF]', ' ', text)  # remove special chars
    return text.strip()

# Build both versions
raw_texts     = [preprocess_raw(a['text'])     for a in all_texts]
cleaned_texts = [preprocess_cleaned(a['text']) for a in all_texts]

# Show sample comparison
print("=== RAW ===")
print(raw_texts[0][:300])
print("\n=== CLEANED ===")
print(cleaned_texts[0][:300])

=== RAW ===
This is a list of lists of universities and colleges by country, sorted by continent and region. The lists represent educational institutions throughout the world which provide higher education in tertiary, quaternary, and post-secondary education.

=== CLEANED ===
this is a list of lists of universities and colleges by country  sorted by continent and region  the lists represent educational institutions throughout the world which provide higher education in tertiary  quaternary  and post secondary education


## Section 4 — Chunking Analysis

Comparing two chunking strategies applied to BOTH raw and cleaned texts:
- **RecursiveCharacterTextSplitter**: respects natural text boundaries
- **TokenTextSplitter**: splits based on token count

In [ ]:
# ── BUILD CHUNKS (only run once, then use load cell below) ──────────────────
'''
from langchain_text_splitters import RecursiveCharacterTextSplitter, TokenTextSplitter

recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500, chunk_overlap=50,
    separators=["\\n\\n", "\\n", ". ", " ", ""]
)
token_splitter = TokenTextSplitter(chunk_size=128, chunk_overlap=16)

recursive_chunks_raw     = []
recursive_chunks_cleaned = []
token_chunks_raw         = []
token_chunks_cleaned     = []

for entry in all_texts:
    raw     = preprocess_raw(entry["text"])
    cleaned = preprocess_cleaned(entry["text"])
    meta    = {"lang": entry["lang"], "title": entry["title"]}

    for chunk in recursive_splitter.split_text(raw):
        recursive_chunks_raw.append({"text": chunk, **meta})
    for chunk in recursive_splitter.split_text(cleaned):
        recursive_chunks_cleaned.append({"text": chunk, **meta})
    for chunk in token_splitter.split_text(raw):
        token_chunks_raw.append({"text": chunk, **meta})
    for chunk in token_splitter.split_text(cleaned):
        token_chunks_cleaned.append({"text": chunk, **meta})

print(f"Recursive RAW     → {len(recursive_chunks_raw)} chunks")
print(f"Recursive CLEANED → {len(recursive_chunks_cleaned)} chunks")
print(f"Token RAW         → {len(token_chunks_raw)} chunks")
print(f"Token CLEANED     → {len(token_chunks_cleaned)} chunks")

# Save all 4 variants to Drive
import json
json.dump([c["text"] for c in recursive_chunks_raw],
    open("/content/drive/MyDrive/chunks_recursive_raw.json",     "w", encoding="utf-8"), ensure_ascii=False)
json.dump([c["text"] for c in recursive_chunks_cleaned],
    open("/content/drive/MyDrive/chunks_recursive_cleaned.json", "w", encoding="utf-8"), ensure_ascii=False)
json.dump([c["text"] for c in token_chunks_raw],
    open("/content/drive/MyDrive/chunks_token_raw.json",         "w", encoding="utf-8"), ensure_ascii=False)
json.dump([c["text"] for c in token_chunks_cleaned],
    open("/content/drive/MyDrive/chunks_token_cleaned.json",     "w", encoding="utf-8"), ensure_ascii=False)
print("All 4 chunk variants saved to Drive!")
'''

'\nfrom langchain_text_splitters import RecursiveCharacterTextSplitter, TokenTextSplitter\n\nrecursive_splitter = RecursiveCharacterTextSplitter(\n    chunk_size=500, chunk_overlap=50,\n    separators=["\\n\\n", "\\n", ". ", " ", ""]\n)\ntoken_splitter = TokenTextSplitter(chunk_size=128, chunk_overlap=16)\n\nrecursive_chunks_raw     = []\nrecursive_chunks_cleaned = []\ntoken_chunks_raw         = []\ntoken_chunks_cleaned     = []\n\nfor entry in all_texts:\n    raw     = preprocess_raw(entry["text"])\n    cleaned = preprocess_cleaned(entry["text"])\n    meta    = {"lang": entry["lang"], "title": entry["title"]}\n\n    for chunk in recursive_splitter.split_text(raw):\n        recursive_chunks_raw.append({"text": chunk, **meta})\n    for chunk in recursive_splitter.split_text(cleaned):\n        recursive_chunks_cleaned.append({"text": chunk, **meta})\n    for chunk in token_splitter.split_text(raw):\n        token_chunks_raw.append({"text": chunk, **meta})\n    for chunk in token_splitter

In [ ]:
# ── LOAD ALL 4 CHUNK VARIANTS FROM DRIVE ─────────────────────────────────────
import json

with open('/content/drive/MyDrive/chunks_recursive_raw.json',     'r', encoding='utf-8') as f:
    recursive_chunks_raw = f.read()
    recursive_chunks_raw = json.loads(recursive_chunks_raw)

with open('/content/drive/MyDrive/chunks_recursive_cleaned.json', 'r', encoding='utf-8') as f:
    recursive_chunks_cleaned = json.load(f)

with open('/content/drive/MyDrive/chunks_token_raw.json',         'r', encoding='utf-8') as f:
    token_chunks_raw = json.load(f)

with open('/content/drive/MyDrive/chunks_token_cleaned.json',     'r', encoding='utf-8') as f:
    token_chunks_cleaned = json.load(f)

print(f"Recursive RAW     → {len(recursive_chunks_raw)} chunks")
print(f"Recursive CLEANED → {len(recursive_chunks_cleaned)} chunks")
print(f"Token RAW         → {len(token_chunks_raw)} chunks")
print(f"Token CLEANED     → {len(token_chunks_cleaned)} chunks")

# Test queries used throughout the notebook
test_queries = [
    ("en", "What is Harvard University known for?"),
    ("de", "Welche Universität ist die beste in Deutschland?"),
    ("es", "¿Cuál es la mejor universidad del mundo?"),
    ("nl", "Welke universiteit is de beste ter wereld?"),
]

Recursive RAW     → 349 chunks
Recursive CLEANED → 305 chunks
Token RAW         → 387 chunks
Token CLEANED     → 388 chunks


In [ ]:
# ── CHUNKING COMPARISON: Independent evaluation per model ─────────────────────
from sentence_transformers import SentenceTransformer
import faiss, numpy as np, json

comparison_queries = [
    "What is the University of Amsterdam known for?",
    "Welche Universität ist die beste in Deutschland?",
    "¿Cuál es la mejor universidad del mundo?",
    "Quelle est la meilleure université du monde?",
]

# ─────────────────────────────────────────────────────────────────────────────
# MODEL 1 — paraphrase-multilingual-MiniLM-L12-v2 | RECURSIVE CHUNKS ONLY
# ─────────────────────────────────────────────────────────────────────────────
model1 = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')

emb1_raw     = model1.encode(recursive_chunks_raw,     show_progress_bar=False)
emb1_cleaned = model1.encode(recursive_chunks_cleaned, show_progress_bar=False)

idx1_raw = faiss.IndexFlatL2(emb1_raw.shape[1])
idx1_raw.add(np.array(emb1_raw))

idx1_cleaned = faiss.IndexFlatL2(emb1_cleaned.shape[1])
idx1_cleaned.add(np.array(emb1_cleaned))

def avg_score_m1(query, index, top_k=3):
    q_emb = model1.encode([query])
    distances, _ = index.search(np.array(q_emb), top_k)
    return float(np.mean(distances[0]))

print("=" * 70)
print("MODEL 1 (MiniLM) — RECURSIVE CHUNKS ONLY")
print("=" * 70)
m1_raw_scores, m1_clean_scores = [], []
for q in comparison_queries:
    s_raw   = avg_score_m1(q, idx1_raw)
    s_clean = avg_score_m1(q, idx1_cleaned)
    m1_raw_scores.append(s_raw)
    m1_clean_scores.append(s_clean)
    print(f"  Query: {q[:60]}")
    print(f"    RAW     score: {s_raw:.4f}")
    print(f"    CLEANED score: {s_clean:.4f}")

overall_m1_raw   = sum(m1_raw_scores)   / len(m1_raw_scores)
overall_m1_clean = sum(m1_clean_scores) / len(m1_clean_scores)
best_recursive_label = "CLEANED" if overall_m1_clean <= overall_m1_raw else "RAW"
best_recursive       = recursive_chunks_cleaned if best_recursive_label == "CLEANED" else recursive_chunks_raw

print(f"\nMODEL 1 Overall → RAW: {overall_m1_raw:.4f} | CLEANED: {overall_m1_clean:.4f}")
print(f"Winner (best_recursive_model1): Recursive {best_recursive_label}")

with open('/content/drive/MyDrive/best_recursive_model1.json', 'w', encoding='utf-8') as f:
    json.dump(best_recursive, f, ensure_ascii=False)

# ─────────────────────────────────────────────────────────────────────────────
# MODEL 2 — intfloat/multilingual-e5-small | TOKEN CHUNKS ONLY
# ─────────────────────────────────────────────────────────────────────────────
model2 = SentenceTransformer('intfloat/multilingual-e5-small')

emb2_raw     = model2.encode(token_chunks_raw,     show_progress_bar=False)
emb2_cleaned = model2.encode(token_chunks_cleaned, show_progress_bar=False)

idx2_raw = faiss.IndexFlatL2(emb2_raw.shape[1])
idx2_raw.add(np.array(emb2_raw))

idx2_cleaned = faiss.IndexFlatL2(emb2_cleaned.shape[1])
idx2_cleaned.add(np.array(emb2_cleaned))

def avg_score_m2(query, index, top_k=3):
    q_emb = model2.encode([query])
    distances, _ = index.search(np.array(q_emb), top_k)
    return float(np.mean(distances[0]))

print("\n" + "=" * 70)
print("MODEL 2 (E5-small) — TOKEN CHUNKS ONLY")
print("=" * 70)
m2_raw_scores, m2_clean_scores = [], []
for q in comparison_queries:
    s_raw   = avg_score_m2(q, idx2_raw)
    s_clean = avg_score_m2(q, idx2_cleaned)
    m2_raw_scores.append(s_raw)
    m2_clean_scores.append(s_clean)
    print(f"  Query: {q[:60]}")
    print(f"    RAW     score: {s_raw:.4f}")
    print(f"    CLEANED score: {s_clean:.4f}")

overall_m2_raw   = sum(m2_raw_scores)   / len(m2_raw_scores)
overall_m2_clean = sum(m2_clean_scores) / len(m2_clean_scores)
best_token_label = "CLEANED" if overall_m2_clean <= overall_m2_raw else "RAW"
best_token       = token_chunks_cleaned if best_token_label == "CLEANED" else token_chunks_raw

print(f"\nMODEL 2 Overall → RAW: {overall_m2_raw:.4f} | CLEANED: {overall_m2_clean:.4f}")
print(f"Winner (best_token_model2): Token {best_token_label}")

with open('/content/drive/MyDrive/best_token_model2.json', 'w', encoding='utf-8') as f:
    json.dump(best_token, f, ensure_ascii=False)

# ─────────────────────────────────────────────────────────────────────────────
# FINAL SUMMARY
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("FINAL STRUCTURE")
print("=" * 70)
print(f"MODEL 1 (MiniLM - Recursive only)")
print(f"  RAW:     {overall_m1_raw:.4f}")
print(f"  CLEANED: {overall_m1_clean:.4f}")
print(f"  WINNER:  {best_recursive_label}")
print(f"MODEL 2 (E5 - Token only)")
print(f"  RAW:     {overall_m2_raw:.4f}")
print(f"  CLEANED: {overall_m2_clean:.4f}")
print(f"  WINNER:  {best_token_label}")

# Set chunk_texts for downstream use (Section 5+ uses Model 1's best recursive)
chunk_texts = best_recursive
print(f"\nchunk_texts set to best recursive ({best_recursive_label}), {len(chunk_texts)} chunks.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MODEL 1 (MiniLM) — RECURSIVE CHUNKS ONLY
  Query: What is the University of Amsterdam known for?
    RAW     score: 21.2444
    CLEANED score: 21.6008
  Query: Welche Universität ist die beste in Deutschland?
    RAW     score: 17.6250
    CLEANED score: 17.6769
  Query: ¿Cuál es la mejor universidad del mundo?
    RAW     score: 10.8403
    CLEANED score: 11.3899
  Query: Quelle est la meilleure université du monde?
    RAW     score: 10.5059
    CLEANED score: 10.8974

MODEL 1 Overall → RAW: 15.0539 | CLEANED: 15.3912
Winner (best_recursive_model1): Recursive RAW


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


MODEL 2 (E5-small) — TOKEN CHUNKS ONLY
  Query: What is the University of Amsterdam known for?
    RAW     score: 0.3877
    CLEANED score: 0.3809
  Query: Welche Universität ist die beste in Deutschland?
    RAW     score: 0.2863
    CLEANED score: 0.2984
  Query: ¿Cuál es la mejor universidad del mundo?
    RAW     score: 0.2324
    CLEANED score: 0.2412
  Query: Quelle est la meilleure université du monde?
    RAW     score: 0.2916
    CLEANED score: 0.2855

MODEL 2 Overall → RAW: 0.2995 | CLEANED: 0.3015
Winner (best_token_model2): Token RAW

FINAL STRUCTURE
MODEL 1 (MiniLM - Recursive only)
  RAW:     15.0539
  CLEANED: 15.3912
  WINNER:  RAW
MODEL 2 (E5 - Token only)
  RAW:     0.2995
  CLEANED: 0.3015
  WINNER:  RAW

chunk_texts set to best recursive (RAW), 349 chunks.


## Section 5 — Retrieval Model 1: `paraphrase-multilingual-MiniLM-L12-v2`

In [ ]:
# ── BUILD & SAVE MODEL 1 INDEX (only run once) ───────────────────────────────
'''
from sentence_transformers import SentenceTransformer
import faiss, numpy as np

retrieval_model_1 = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
print("Retrieval Model 1 loaded.")

print("Embedding chunks...")
embeddings_1 = retrieval_model_1.encode(chunk_texts, show_progress_bar=True)
index_1 = faiss.IndexFlatL2(embeddings_1.shape[1])
index_1.add(np.array(embeddings_1))

faiss.write_index(index_1, "/content/drive/MyDrive/uni_index_model1.faiss")
np.save("/content/drive/MyDrive/uni_embeddings_model1.npy", embeddings_1)
print("Model 1 index saved to Drive!")
'''

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Retrieval Model 1 loaded.
Embedding chunks...


Batches:   0%|          | 0/11 [00:00<?, ?it/s]

Model 1 index saved to Drive!


In [ ]:
# ── LOAD MODEL 1 INDEX FROM DRIVE ────────────────────────────────────────────
from sentence_transformers import SentenceTransformer
import faiss, numpy as np

retrieval_model_1 = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')
index_1      = faiss.read_index('/content/drive/MyDrive/uni_index_model1.faiss')
embeddings_1 = np.load('/content/drive/MyDrive/uni_embeddings_model1.npy')
print("Model 1 loaded and index ready!")

def retrieve_model1(query, top_k=3):
    q_emb              = retrieval_model_1.encode([query])
    distances, indices = index_1.search(np.array(q_emb), top_k)
    return [(chunk_texts[i], float(distances[0][j])) for j, i in enumerate(indices[0])]

# Quick test
print("\n=== MODEL 1 QUICK TEST ===")
for lang, query in test_queries:
    results = retrieve_model1(query)
    print(f"[{lang}] {query}")
    print(f"  Top result: {results[0][0][:120]}...\n")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Model 1 loaded and index ready!

=== MODEL 1 QUICK TEST ===
[en] What is Harvard University known for?
  Top result: . Den Gegenbegriff zur „Massenuniversität“ bildet die „Eliteuniversität“ (siehe: Bildungschance, Spitzenuniversität)....

[de] Welche Universität ist die beste in Deutschland?
  Top result: Liste der Hochschulen in Deutschland
FernUniversität in Hagen
Open University...

[es] ¿Cuál es la mejor universidad del mundo?
  Top result: . Den Gegenbegriff zur „Massenuniversität“ bildet die „Eliteuniversität“ (siehe: Bildungschance, Spitzenuniversität)....

[nl] Welke universiteit is de beste ter wereld?
  Top result: Esta lista contiene las universidades más grandes del mundo por número de alumnos matriculados....



## Section 6 — Retrieval Model 2: `intfloat/multilingual-e5-small`

E5 requires `passage:` prefix on chunks during indexing and `query:` prefix on queries during search.

In [ ]:
# ── BUILD & SAVE MODEL 2 INDEX (only run once) ───────────────────────────────
'''
retrieval_model_2 = SentenceTransformer("intfloat/multilingual-e5-small")
print("Retrieval Model 2 loaded.")

e5_chunk_texts = ["passage: " + c for c in best_token]
print("Embedding chunks with E5...")
embeddings_2 = retrieval_model_2.encode(e5_chunk_texts, show_progress_bar=True)
index_2 = faiss.IndexFlatL2(embeddings_2.shape[1])
index_2.add(np.array(embeddings_2))

faiss.write_index(index_2, "/content/drive/MyDrive/uni_index_model2.faiss")
np.save("/content/drive/MyDrive/uni_embeddings_model2.npy", embeddings_2)
print("Model 2 index saved to Drive!")
'''

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Retrieval Model 2 loaded.
Embedding chunks with E5...


Batches:   0%|          | 0/13 [00:00<?, ?it/s]

Model 2 index saved to Drive!


In [ ]:
# ── LOAD MODEL 2 INDEX FROM DRIVE ────────────────────────────────────────────
retrieval_model_2 = SentenceTransformer('intfloat/multilingual-e5-small')
index_2      = faiss.read_index('/content/drive/MyDrive/uni_index_model2.faiss')
embeddings_2 = np.load('/content/drive/MyDrive/uni_embeddings_model2.npy')
print("Model 2 loaded and index ready!")

def retrieve_model2(query, top_k=3):
    q_emb              = retrieval_model_2.encode(["query: " + query])
    distances, indices = index_2.search(np.array(q_emb), top_k)
    return [(best_token[i], float(distances[0][j])) for j, i in enumerate(indices[0])]

# Quick test
print("\n=== MODEL 2 QUICK TEST ===")
for lang, query in test_queries:
    results = retrieve_model2(query)
    print(f"[{lang}] {query}")
    print(f"  Top result: {results[0][0][:120]}...\n")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Model 2 loaded and index ready!

=== MODEL 2 QUICK TEST ===
[en] What is Harvard University known for?
  Top result: De Vereniging van Afrikaanse Universiteiten (Engels: Association of African Universities, Frans: Association des univers...

[de] Welche Universität ist die beste in Deutschland?
  Top result:  Hochschulen in Deutschland
FernUniversität in Hagen
Open University...

[es] ¿Cuál es la mejor universidad del mundo?
  Top result: Esta lista contiene las universidades más grandes del mundo por número de alumnos matriculados....

[nl] Welke universiteit is de beste ter wereld?
  Top result: De Universiteit van Damascus (Arabisch: جامعة دمشق, Jāmi'atu Dimashq) is de grootste en oudste universiteit in Syrië. Zi...



## Section 7 — Retrieval Model Comparison

Same queries through both models. Lower L2 score = better retrieval.

In [ ]:
import time

print("=" * 70)
print("RETRIEVAL MODEL COMPARISON")
print("=" * 70)

for lang, query in test_queries:
    print(f"\nQuery [{lang}]: {query}")

    t1   = time.time()
    res1 = retrieve_model1(query, top_k=1)
    time1 = time.time() - t1

    t2   = time.time()
    res2 = retrieve_model2(query, top_k=1)
    time2 = time.time() - t2

    print(f"  Model 1 (MiniLM) [{time1:.3f}s] score={res1[0][1]:.4f}: {res1[0][0][:100]}...")
    print(f"  Model 2 (E5)     [{time2:.3f}s] score={res2[0][1]:.4f}: {res2[0][0][:100]}...")
    print("-" * 70)

print("\nLower score = more similar result. Compare chunk relevance manually.")

RETRIEVAL MODEL COMPARISON

Query [en]: What is Harvard University known for?
  Model 1 (MiniLM) [0.014s] score=21.1549: . Den Gegenbegriff zur „Massenuniversität“ bildet die „Eliteuniversität“ (siehe: Bildungschance, Spi...
  Model 2 (E5)     [0.011s] score=0.4537: De Vereniging van Afrikaanse Universiteiten (Engels: Association of African Universities, Frans: Ass...
----------------------------------------------------------------------

Query [de]: Welche Universität ist die beste in Deutschland?
  Model 1 (MiniLM) [0.010s] score=16.0460: Liste der Hochschulen in Deutschland
FernUniversität in Hagen
Open University...
  Model 2 (E5)     [0.011s] score=0.3078:  Hochschulen in Deutschland
FernUniversität in Hagen
Open University...
----------------------------------------------------------------------

Query [es]: ¿Cuál es la mejor universidad del mundo?
  Model 1 (MiniLM) [0.011s] score=10.3576: . Den Gegenbegriff zur „Massenuniversität“ bildet die „Eliteuniversität“ (siehe: Bildungsc

## Section 8 — Generation Model 1: `google/gemma-3-4b-it`

In [ ]:
# ── DOWNLOAD FROM HUGGINGFACE & SAVE TO DRIVE (only run once) ───────────────
'''
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16
)
tokenizer = AutoTokenizer.from_pretrained("google/gemma-3-4b-it", token=hf_token)
model     = AutoModelForCausalLM.from_pretrained(
    "google/gemma-3-4b-it", token=hf_token,
    quantization_config=bnb_config, device_map="auto"
)
tokenizer.save_pretrained("/content/drive/MyDrive/gemma-3-4b-it")
model.save_pretrained("/content/drive/MyDrive/gemma-3-4b-it")
print("Gemma downloaded and saved to Drive!")
'''

'\nfrom transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig\nimport torch\n\nbnb_config = BitsAndBytesConfig(\n    load_in_4bit=True, bnb_4bit_use_double_quant=True,\n    bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16\n)\ntokenizer = AutoTokenizer.from_pretrained("google/gemma-3-4b-it", token=hf_token)\nmodel     = AutoModelForCausalLM.from_pretrained(\n    "google/gemma-3-4b-it", token=hf_token,\n    quantization_config=bnb_config, device_map="auto"\n)\ntokenizer.save_pretrained("/content/drive/MyDrive/gemma-3-4b-it")\nmodel.save_pretrained("/content/drive/MyDrive/gemma-3-4b-it")\nprint("Gemma downloaded and saved to Drive!")\n'

In [ ]:
# ── LOAD GEMMA FROM DRIVE ────────────────────────────────────────────────────
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16
)
save_path = '/content/drive/MyDrive/gemma-3-4b-it'
tokenizer = AutoTokenizer.from_pretrained(save_path)
model     = AutoModelForCausalLM.from_pretrained(
    save_path, quantization_config=bnb_config, device_map="auto"
)
print(f"Gemma loaded from Drive!")
print(f"Device: {next(model.parameters()).device}")

/usr/local/lib/python3.12/dist-packages/transformers/quantizers/auto.py:271: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)


Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

Gemma loaded from Drive!
Device: cuda:0


## Section 9 — Generation Model 2: `Qwen/Qwen2.5-3B-Instruct`

**Free GPU memory before loading if Gemma is still in memory.**

In [ ]:
# ── FREE GPU MEMORY (run if Gemma is loaded and you want to switch) ─────────
'''
import gc, torch
del model
del tokenizer
gc.collect()
torch.cuda.empty_cache()
print("GPU memory freed!")
'''

In [ ]:
# ── DOWNLOAD QWEN FROM HUGGINGFACE & SAVE TO DRIVE (only run once) ──────────
'''
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16
)
tokenizer2 = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-3B-Instruct", token=hf_token)
model2     = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-3B-Instruct", token=hf_token,
    quantization_config=bnb_config, device_map="auto"
)
tokenizer2.save_pretrained("/content/drive/MyDrive/qwen2.5-3b-instruct")
model2.save_pretrained("/content/drive/MyDrive/qwen2.5-3b-instruct")
print("Qwen downloaded and saved to Drive!")
'''

In [ ]:
# ── LOAD QWEN FROM DRIVE ─────────────────────────────────────────────────────
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16
)
save_path2 = '/content/drive/MyDrive/qwen2.5-3b-instruct'
tokenizer2 = AutoTokenizer.from_pretrained(save_path2)
model2     = AutoModelForCausalLM.from_pretrained(
    save_path2, quantization_config=bnb_config, device_map="auto"
)
print("Qwen loaded from Drive!")
print(f"Device: {next(model2.parameters()).device}")

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Qwen loaded from Drive!
Device: cuda:0


## Section 10 — Full RAG Pipeline with Language Control

- **Auto mode**: detects language from query automatically
- **Override mode**: pass `output_lang='French'` to force any output language

In [ ]:
import time

# ── Language detection ────────────────────────────────────────────────────────
def detect_language(query):
    if any(w in query.lower() for w in ['cual', 'qué', 'cómo', 'universidad', 'es la']):
        return "Spanish"
    elif any(w in query.lower() for w in ['welche', 'welcher', 'universität', 'beste', 'die']):
        return "German"
    elif any(w in query.lower() for w in ['welke', 'ter wereld', 'universiteit', 'de beste']):
        return "Dutch"
    else:
        return "English"

# ── Core generation function ──────────────────────────────────────────────────
def generate_response(prompt, tok, mod):
    messages = [{"role": "user", "content": prompt}]
    text     = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs   = tok(text, return_tensors="pt").to(mod.device)
    outputs  = mod.generate(
        **inputs, max_new_tokens=200, do_sample=True,
        temperature=0.7, pad_token_id=tok.eos_token_id
    )
    return tok.decode(outputs[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True).strip()

# ── Full RAG pipeline ─────────────────────────────────────────────────────────
def rag_pipeline(query, tok, mod, retrieval_fn=None, output_lang=None):
    """
    Args:
        query        : user question in any language
        tok          : tokenizer (tokenizer for Gemma, tokenizer2 for Qwen)
        mod          : model    (model for Gemma, model2 for Qwen)
        retrieval_fn : retrieve_model1 or retrieve_model2 (defaults to model1)
        output_lang  : force output language e.g. 'German', 'Spanish'.
                       If None, auto-detected from query.
    """
    if retrieval_fn is None:
        retrieval_fn = retrieve_model1

    # Step 1: Retrieve
    retrieved = retrieval_fn(query, top_k=3)
    context   = "\n\n".join([chunk for chunk, _ in retrieved])

    # Step 2: Determine output language
    lang = output_lang if output_lang else detect_language(query)

    # Step 3: Build prompt with explicit language instruction
    prompt = (
        f"You are a multilingual university information assistant. "
        f"Answer using ONLY the context below. "
        f"You MUST respond in {lang}, regardless of the language of the question or context.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {query}"
    )

    # Step 4: Generate
    return generate_response(prompt, tok, mod)

print("RAG pipeline ready.")

RAG pipeline ready.


In [ ]:
# ── TEST: Auto language detection ────────────────────────────────────────────
print("=== RAG PIPELINE — Auto Language Detection ===")

for lang, query in test_queries:
    detected = detect_language(query)
    print(f"\n[{lang}] Query    : {query}")
    print(f"       Detected : {detected}")
    start  = time.time()
    answer = rag_pipeline(query, tokenizer, model)
    print(f"       Answer   : {answer}")
    print(f"       Time     : {time.time()-start:.2f}s")
    print("-" * 60)

=== RAG PIPELINE — Auto Language Detection ===

[en] Query    : What is Harvard University known for?
       Detected : English
       Answer   : According to the text, Harvard University was founded in Cambridge, USA in 1636.
       Time     : 2.95s
------------------------------------------------------------

[de] Query    : Welche Universität ist die beste in Deutschland?
       Detected : German
       Answer   : Basierend auf dem gegebenen Text ist die Universität Heidelberg (1386) die älteste Universität in Deutschland. Der Text gibt jedoch keine Auskunft darüber, welche Universität die „beste“ ist.
       Time     : 5.83s
------------------------------------------------------------

[es] Query    : ¿Cuál es la mejor universidad del mundo?
       Detected : Spanish
       Answer   : El texto proporcionado no ofrece información sobre cuál es la "mejor" universidad del mundo.  Describe la existencia de universidades en la Corona de Aragón (Valencia y Barcelona en el siglo XIV) y di

In [ ]:
# ── TEST: Language override ───────────────────────────────────────────────────
print("=== RAG PIPELINE — Language Override ===")

override_examples = [
    ("Welche Universität ist die beste in Deutschland?", "Dutch"),   # German query → Dutch output
    ("What is the best university in the world?",        "Spanish"),    # English query → Spanish output
    ("¿Cuál es la mejor universidad del mundo?",         "English"), # Spanish query → English output
]

for query, forced_lang in override_examples:
    print(f"\nQuery      : {query}")
    print(f"Forced lang: {forced_lang}")
    start  = time.time()
    answer = rag_pipeline(query, tokenizer, model, output_lang=forced_lang)
    print(f"Answer     : {answer}")
    print(f"Time       : {time.time()-start:.2f}s")
    print("-" * 60)

=== RAG PIPELINE — Language Override ===

Query      : Welche Universität ist die beste in Deutschland?
Forced lang: Dutch
Answer     : De context vermeldt dat de Universiteit Heidelberg de oudste is in het huidige Duitsland. Er wordt niet gesproken over welke universiteit de "beste" is.
Time       : 4.75s
------------------------------------------------------------

Query      : What is the best university in the world?
Forced lang: Spanish
Answer     : El texto proporcionado no ofrece información sobre qué universidad es la mejor del mundo. Describe la existencia de universidades en la Corona de Aragón y la diferencia entre universidades presenciales y virtuales.
Time       : 5.61s
------------------------------------------------------------

Query      : ¿Cuál es la mejor universidad del mundo?
Forced lang: English
Answer     : The provided text does not contain information about which university is the “best in the world.” It discusses the contrast between human capabilities and ma

## Section 11 — Generation Model Comparison

Same query + same context through both models. **Load Qwen in Section 9 before running this.**

In [ ]:
print("=" * 70)
print("GENERATION MODEL COMPARISON")
print("=" * 70)

gen_comparison = [
    ("en", "What is a university campus?",                     None),
    ("de", "Welche Universität ist die beste in Deutschland?", None),
    ("es", "¿Cuál es la mejor universidad del mundo?",         None),
    ("nl", "Welke universiteit is de beste ter wereld?",        None),
    ("override", "What is the best university in the world?",  "Dutch"),
]
for lang, query, forced_lang in gen_comparison:
    print(f"\nQuery [{lang}]: {query}")
    if forced_lang:
        print(f"  Forced output: {forced_lang}")

    t1   = time.time()
    ans1 = rag_pipeline(query, tokenizer, model, output_lang=forced_lang)
    time1 = time.time() - t1

    try:
        t2   = time.time()
        ans2 = rag_pipeline(query, tokenizer2, model2, output_lang=forced_lang)
        time2 = time.time() - t2
    except NameError:
        ans2  = "[Qwen not loaded — run Section 9 load cell first]"
        time2 = 0

    print(f"  Gemma   [{time1:.2f}s]: {ans1[:200]}")
    print(f"  Qwen    [{time2:.2f}s]: {ans2[:200]}")
    print("-" * 70)

GENERATION MODEL COMPARISON

Query [en]: What is a university campus?
  Gemma   [7.41s]: The provided text does not contain information about what a university campus is. It discusses historical universities in Valencia and Barcelona, the "quodlibet" method of questioning in medieval educ
  Qwen    [7.26s]: A university campus refers to the physical location where a university operates, including its buildings, facilities, and grounds. It's the place where students attend classes, laboratories, libraries
----------------------------------------------------------------------

Query [de]: Welche Universität ist die beste in Deutschland?
  Gemma   [9.19s]: Basierend auf dem bereitgestellten Kontext ist es nicht möglich zu sagen, welche Universität in Deutschland die "beste" ist. Der Text erwähnt lediglich die Universität Heidelberg (1386) als die ältest
  Qwen    [5.30s]: Die Universität Heidelberg ist die älteste und oft angesehenste Universität in Deutschland. Sie wurde 1386 gegründet u

## Section 12 — Edge Case Testing

### Test 1 — Within Dataset (University in dataset)

In [ ]:
# Query about a university that IS in the dataset
# Expected: good grounded answer from retrieved context
query = "What is the University of Amsterdam known for?"
print(f"Query: {query}")
print(f"Detected language: {detect_language(query)}")
answer = rag_pipeline(query, tokenizer, model)
print(f"Answer: {answer}")

Query: What is the University of Amsterdam known for?
Detected language: English
Answer: The provided text does not contain any information about the University of Amsterdam. It describes adjunct professors and anatomical theaters, specifically mentioning the University of Padua and the University of Leiden.


### Test 2 — Outside Dataset (Hallucination test)

In [ ]:
# Query about a university NOT in the dataset (MAJU, Karachi)
# Expected: model either admits it doesn't know, or hallucinate
query = "What is MAJU university in Karachi known for?"
print(f"Query: {query}")
retrieved = retrieve_model1(query)
print("Retrieved chunks (likely irrelevant):")
for i, (chunk, score) in enumerate(retrieved):
    print(f"  [{i+1}] score={score:.4f}: {chunk[:100]}...")
print()
answer = rag_pipeline(query, tokenizer, model)
print(f"Answer: {answer}")

Query: What is MAJU university in Karachi known for?
Retrieved chunks (likely irrelevant):
  [1] score=15.5651: Een decaan (Engels: dean, Frans: doyen, Duits: Dekan) (niet te verwarren met decaan in het voortgeze...
  [2] score=16.5765: Licentiaat is een oude academische graad die verschillende onderwijsniveaus in verschillende landen ...
  [3] score=16.9664: . Die verschiedenen Standorte unterscheiden sich hinsichtlich ihrer fachlichen Ausrichtung....

Answer: The provided text does not contain any information about MAJU university in Karachi. It discusses the role of a “decan” (dean) at a university and the academic degree “licentiaat.”


### Test 3 — Language Output Control

In [ ]:
# Ask in one language, force output in another
lang_control_tests = [
    ("What is the best university in Germany?",          "Dutch"),
    ("Welche Universität ist die beste?",                "Spanish"),
    ("Welke universiteit is de beste ter wereld?",       "English"),
]

for query, forced_lang in lang_control_tests:
    print(f"Query       : {query}")
    print(f"Forced lang : {forced_lang}")
    answer = rag_pipeline(query, tokenizer, model, output_lang=forced_lang)
    print(f"Answer      : {answer}")
    print("-" * 60)

Query       : What is the best university in Germany?
Forced lang : Dutch
Answer      : Volgens de verstrekte informatie is de Universität Heidelberg (1386) de oudste universiteit in het huidige Duitsland.
------------------------------------------------------------
Query       : Welche Universität ist die beste?
Forced lang : Spanish
Answer      : El texto describe dos tipos de universidades: la *Präsenzuniversität* (universidad presencial) y la *Fernuniversität* (universidad a distancia).  No evalúa cuál es "la mejor" universidad, solo presenta sus diferencias.  El texto se centra en la importancia del ser humano y sus habilidades para resolver problemas y transformar la realidad, en contraste con la capacidad de las máquinas.
------------------------------------------------------------
Query       : Welke universiteit is de beste ter wereld?
Forced lang : English
Answer      : The context provided discusses the concept of “presence universities” which are universities where lectures

### Test 4 — Unsupported Language

In [ ]:
# Force output in a language the model may not know well
# Swahili: small but real presence in training data
# Punjabi: very low representation
unsupported_tests = [
    ("What is a university?", "Swahili"),
    ("What is a university?", "Punjabi"),
]

for query, forced_lang in unsupported_tests:
    print(f"Query       : {query}")
    print(f"Forced lang : {forced_lang}")
    answer = rag_pipeline(query, tokenizer, model, output_lang=forced_lang)
    print(f"Answer      : {answer}")
    print("-" * 60)

Query       : What is a university?
Forced lang : Swahili
Answer      : Kisha, shule ya juu ni taasara inayofaa masaa ya utafiti na elimu, na inalenga kutoa vyezi vya ubinadamu. (A university is an institution dedicated to academic study and education, aiming to provide human knowledge.)
------------------------------------------------------------
Query       : What is a university?
Forced lang : Punjabi
Answer      : ਸੰਪੂਰਨ ਵਿਦਿਅਕ ਸੰਸਥਾ ਇੱਕ ਅਜਿਹੀ ਥਾਂ ਹੈ ਜਿੱਥੇ ਗਿਆਨ ਅਤੇ ਵਿਦਿਅ ਦਾ ਅਧਿਆਨ ਕੀਤਾ ਜਾਂਦਾ ਹੈ। ਇਹ ਵਿਦਿਅਕ ਸੰਸਥਾ ਇੱਕ ਸਕੂਲ ਜਾਂ ਯੂਨੀਵਰਸਿਟੀ ਹੋ ਸਕਦੀ ਹੈ।
------------------------------------------------------------


In [ ]:
# Force output in a language the model may not know well
# Swahili: small but real presence in training data
# Punjabi: very low representation
unsupported_tests = [
    # Query in unsupported language → force English output
    ("Chuo kikuu ni nini?",   "English"),  # Swahili input → English output
    ("ਯੂਨੀਵਰਸਿਟੀ ਕੀ ਹੈ?",     "English"),  # Punjabi input → English output
]

for query, forced_lang in unsupported_tests:
    print(f"Query       : {query}")
    print(f"Forced lang : {forced_lang}")
    answer = rag_pipeline(query, tokenizer, model, output_lang=forced_lang)
    print(f"Answer      : {answer}")
    print("-" * 60)

Query       : Chuo kikuu ni nini?
Forced lang : English
Answer      : The context does not provide information about "Chuo kikuu." It focuses on the structure of university governance in the Netherlands, specifically mentioning the College of Deans and the Rector Magnificus.
------------------------------------------------------------
Query       : ਯੂਨੀਵਰਸਿਟੀ ਕੀ ਹੈ?
Forced lang : English
Answer      : The Caribbean Medical University (CMU) is a university located in Willemstad, Curaçao. It offers an American curriculum and trains students for the Doctor of Medicine (MD) degree.
------------------------------------------------------------
